In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
from pymc.model import Model
import os
import glob

In [ ]:
def process_mrt_data(df):

  df = df.query('viability == 1')
  df['decision_date'] = pd.to_datetime(df['decision_time']).dt.date

  df['user_start_day_dt'] = pd.to_datetime(df['user_start_day']).dt.date
  df['decision_date'] = pd.to_datetime(df['decision_date'])
  df['user_start_day_dt'] = pd.to_datetime(df['user_start_day'])

  df['state_day_type'] = pd.to_datetime(df['decision_time']).apply(lambda x: 1 if x.weekday() >= 5 else 0)

  df = df.drop(columns = ['state_modif'])

  df['state_day_in_study'] = (df['decision_date'] - df['user_start_day_dt']).dt.days + 1
  df['state_day_in_study'] = (df['state_day_in_study'] - 35.5) / 34.5

  desired_order = ['user_id', 'decision_time', 'action', 'quality', 'state_tod',
                 'state_b_bar', 'state_a_bar', 'state_app_engage', 'state_day_type',
                 'state_bias', 'state_day_in_study']
  df = df[desired_order]
  return df

def get_user_data(df, user_id):
  return df[df['user_id'] == user_id]


In [32]:
script_dir = os.getcwd()
mrt_data_path = os.path.join(script_dir, '../../../OralyticsMRT/oralytics_mrt_data.csv')

MRT_DATA = pd.read_csv(mrt_data_path)
MRT_DATA = process_mrt_data(MRT_DATA)
MRT_USERS = MRT_DATA['user_id'].unique().tolist()

/var/folders/hz/q4hdsnpj1h50x_jpx8v5mrvc0000gn/T/ipykernel_58430/3820211983.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['decision_date'] = pd.to_datetime(df['decision_time']).dt.date
/var/folders/hz/q4hdsnpj1h50x_jpx8v5mrvc0000gn/T/ipykernel_58430/3820211983.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['user_start_day_dt'] = pd.to_datetime(df['user_start_day']).dt.date
/var/folders/hz/q4hdsnpj1h50x_jpx8v5mrvc0000gn/T/ipykernel_58430/3820211983.py:7: SettingWithCopyWarning: 
A value is t

In [ ]:
def merge_message_types(mrt_data, message_type_dir):
    """
    Merge message_type data from individual user CSV files into MRT_DATA.
    
    Args:
        mrt_data: DataFrame with MRT data
        message_type_dir: Directory containing user-specific CSV files
    
    Returns:
        DataFrame with message_type column added
    """
    
    # Get all CSV files in the message type directory
    csv_files = glob.glob(os.path.join(message_type_dir, "*.csv"))
    
    # List to store all message type data
    all_message_data = []
    
    print(f"Found {len(csv_files)} message type files")
    
    for file_path in csv_files:
        # Extract user_id from filename (remove .csv extension)
        user_id = os.path.basename(file_path).replace('.csv', '')
        

        # Read the user's message type data
        user_message_data = pd.read_csv(file_path)
        
        # Add user_id column
        user_message_data['user_id'] = user_id
        
        # Select only the columns we need for merging
        merge_cols = ['user_id', 'decision_time', 'message_type']
        user_message_data = user_message_data[merge_cols]
        
        all_message_data.append(user_message_data)
            

    
    # Combine all message type data

    message_type_df = pd.concat(all_message_data, ignore_index=True)
    
    # Convert decision_time to datetime for both DataFrames
    mrt_data_copy = mrt_data.copy()
    mrt_data_copy['decision_time'] = pd.to_datetime(mrt_data_copy['decision_time'])
    message_type_df['decision_time'] = pd.to_datetime(message_type_df['decision_time'])
    
    # Merge on user_id and decision_time
    merged_data = mrt_data_copy.merge(
        message_type_df, 
        on=['user_id', 'decision_time'], 
        how='left'
    )
    
    print(f"Original MRT_DATA shape: {mrt_data.shape}")
    print(f"Message type data shape: {message_type_df.shape}")
    print(f"Merged data shape: {merged_data.shape}")
    print(f"Rows with message_type: {merged_data['message_type'].notna().sum()}")
    
    return merged_data



In [ ]:
# Merge message type data with MRT_DATA
message_type_path = os.path.join(script_dir, '../../../OralyticsMRT/MRT_messagetype_parsing/message_type_df/')

# Apply the merge function
MRT_DATA_WITH_MESSAGE_TYPE = merge_message_types(MRT_DATA, message_type_path)


In [ ]:
# Create simplified message type categories
MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'] = MRT_DATA_WITH_MESSAGE_TYPE['message_type'].str.replace(r'-\d{2}$', '', regex=True)

# Check the distribution
print("Message type category distribution:")
print(MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'].value_counts(dropna=False))
print(MRT_DATA_WITH_MESSAGE_TYPE.head())

In [ ]:
def get_batch_data(df, user_id):
  user_df = get_user_data(df, user_id)
  states_df = user_df.filter(regex='state_*')
  outcomes = user_df['quality']
  actions = user_df['message_type_category']

  return np.array(states_df), np.array(outcomes), np.array(actions)

In [ ]:
# Check for rows with action = 1.0 but message_type is NaN
action_1_no_message = MRT_DATA_WITH_MESSAGE_TYPE[(MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0) & (MRT_DATA_WITH_MESSAGE_TYPE['message_type'].isna())]

print(f"Rows with action=1.0 but message_type is NaN: {len(action_1_no_message)}")
print(f"Total rows with action=1.0: {len(MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0])}")
print(f"Percentage of action=1.0 rows missing message_type: {len(action_1_no_message) / len(MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0]) * 100:.1f}%")

if len(action_1_no_message) > 0:
    print(f"\nFirst few examples:")
    print(action_1_no_message[['user_id', 'decision_time', 'action', 'message_type']].head())


## Fitting Models
---

### Helpers

In [ ]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

def build_zip_model(X, A, Y):
  model = pm.Model()
  with Model() as model:
    d = X.shape[1]
    
    # Baseline parameters (when A is NaN)
    w_b = pm.MvNormal('w_b', mu=np.zeros(d), cov=np.eye(d), shape=d)
    w_p = pm.MvNormal('w_p', mu=np.zeros(d), cov=np.eye(d), shape=d)
    
    # Categorical effects for each message type (excluding NaN baseline)
    # Get unique categories excluding NaN
    A_series = pd.Series(A)
    categories = [cat for cat in A_series.unique() if pd.notna(cat)]
    n_categories = len(categories)
    
    # Effects for each category
    delta_b = pm.MvNormal('delta_b', mu=np.zeros(d), cov=np.eye(d), shape=(n_categories, d))
    delta_p = pm.MvNormal('delta_p', mu=np.zeros(d), cov=np.eye(d), shape=(n_categories, d))
    
    # Create design matrix for categorical effects
    bern_term = X @ w_b
    poisson_term = X @ w_p
    
    # Add categorical effects
    for i, category in enumerate(categories):
        mask = (A == category)
        if mask.any():
            bern_term = bern_term + mask * (X @ delta_b[i])
            poisson_term = poisson_term + mask * (X @ delta_p[i])
    
    R = pm.ZeroInflatedPoisson("likelihood", psi=1 - sigmoid(bern_term), mu=np.exp(poisson_term), observed=Y)

  return model

def run_zip_map_for_users(users_states, users_actions, users_rewards, num_restarts):
  model_params = {}

  for user_id in users_states.keys():
    print("FOR USER: ", user_id)
    user_states = users_states[user_id]
    d = user_states.shape[1]
    user_actions = users_actions[user_id]
    user_rewards = users_rewards[user_id]
    
    # Get number of categories (excluding NaN)
    # Convert to pandas Series to handle mixed types properly
    user_actions_series = pd.Series(user_actions)
    categories = [cat for cat in user_actions_series.unique() if pd.notna(cat)]
    n_categories = len(categories)
    
    logp_vals = np.empty(shape=(num_restarts,))
    # Calculate total parameter size: w_b(d) + delta_b(n_cat*d) + w_p(d) + delta_p(n_cat*d)
    total_params = d + n_categories * d + d + n_categories * d
    param_vals = np.empty(shape=(num_restarts, total_params))
    
    for seed in range(num_restarts):
      model = build_zip_model(user_states, user_actions, user_rewards)
      np.random.seed(seed)
      
      # Initialize parameters with correct shapes
      init_params = {
        'w_b': np.random.randn(d), 
        'delta_b': np.random.randn(n_categories, d), 
        'w_p': np.random.randn(d), 
        'delta_p': np.random.randn(n_categories, d)
      }
      
      with model:
        map_estimate = pm.find_MAP(start=init_params)

      w_b = map_estimate['w_b']
      delta_b = map_estimate['delta_b']
      w_p = map_estimate['w_p']
      delta_p = map_estimate['delta_p']
      logp_vals[seed] = model.compile_logp()(map_estimate)
      param_vals[seed] = np.concatenate((w_b, delta_b.flatten(), w_p, delta_p.flatten()), axis=None)
    model_params[user_id] = param_vals[np.argmax(logp_vals)]

  return model_params

### Execution

In [ ]:
users_states = {}
users_rewards = {}
users_actions = {}
for user_id in MRT_USERS:
    states, rewards, actions = get_batch_data(MRT_DATA_WITH_MESSAGE_TYPE, user_id)
    users_rewards[user_id] = rewards
    users_actions[user_id] = actions
    users_states[user_id] = states

In [ ]:
zip_model_params = run_zip_map_for_users(users_states, users_actions, users_rewards, num_restarts=5)

## Saving Parameter Values
---

In [23]:
def create_model_columns(feature_names, message_categories):
    """
    Create column names for the categorical ZIP model parameters.
    
    Args:
        feature_names: List of feature names (e.g., ['state_tod', 'state_b_bar', ...])
        message_categories: List of message type categories (e.g., ['QA', 'SR', ...])
    
    Returns:
        List of column names
    """
    columns = ['User']
    
    # Baseline parameters (Bernoulli and Poisson)
    for feature in feature_names:
        columns.append(f'{feature}.Base.Bern')
    for feature in feature_names:
        columns.append(f'{feature}.Base.Poisson')
    
    # Category-specific parameters for each message type
    for category in message_categories:
        for feature in feature_names:
            columns.append(f'{feature}.{category}.Bern')
        for feature in feature_names:
            columns.append(f'{feature}.{category}.Poisson')
    
    return columns

# Get unique message categories from the data (excluding NaN)
message_categories = [cat for cat in MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'].unique() if pd.notna(cat)]
feature_names = ['state_tod', 'state_b_bar.norm', 'state_a_bar.norm', 'state_app_engage', 
                 'state_day_type', 'state_bias', 'state_day_in_study']

# Create the column names dynamically
non_stat_zip_model_columns = create_model_columns(feature_names, message_categories)

print(f"Message categories found: {message_categories}")
print(f"Number of columns: {len(non_stat_zip_model_columns)}")
print("First 10 columns:", non_stat_zip_model_columns[:10])


Message categories found: ['RP', 'SR', 'QA', 'FB']
Number of columns: 71
First 10 columns: ['User', 'state_tod.Base.Bern', 'state_b_bar.norm.Base.Bern', 'state_a_bar.norm.Base.Bern', 'state_app_engage.Base.Bern', 'state_day_type.Base.Bern', 'state_bias.Base.Bern', 'state_day_in_study.Base.Bern', 'state_tod.Base.Poisson', 'state_b_bar.norm.Base.Poisson']


In [28]:
def fill_missing_parameters_with_averages(zip_model_params, non_stat_zip_model_columns, feature_names, message_categories):
    """
    Correctly fill missing parameters by mapping them to the right columns
    based on the user's actual message type categories.
    """
    import numpy as np
    
    rows = []
    
    for user in zip_model_params.keys():
        values = zip_model_params[user]
        new_row = {'User': user}
        
        # Get this user's actual message type categories
        user_data = MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user]
        user_categories = [cat for cat in user_data['message_type_category'].unique() if pd.notna(cat)]
        
        # Map parameters to columns correctly
        param_idx = 0
        
        # 1. Baseline parameters (w_b and w_p)
        for feature in feature_names:
            if param_idx < len(values):
                new_row[f'{feature}.Base.Bern'] = values[param_idx]
                param_idx += 1
            else:
                new_row[f'{feature}.Base.Bern'] = np.nan
                
        for feature in feature_names:
            if param_idx < len(values):
                new_row[f'{feature}.Base.Poisson'] = values[param_idx]
                param_idx += 1
            else:
                new_row[f'{feature}.Base.Poisson'] = np.nan
        
        # 2. Category-specific parameters (delta_b and delta_p)
        for category in message_categories:
            if category in user_categories:
                # User has this category, use their parameters
                for feature in feature_names:
                    if param_idx < len(values):
                        new_row[f'{feature}.{category}.Bern'] = values[param_idx]
                        param_idx += 1
                    else:
                        new_row[f'{feature}.{category}.Bern'] = np.nan
                        
                for feature in feature_names:
                    if param_idx < len(values):
                        new_row[f'{feature}.{category}.Poisson'] = values[param_idx]
                        param_idx += 1
                    else:
                        new_row[f'{feature}.{category}.Poisson'] = np.nan
            else:
                # User doesn't have this category, will fill with average later
                for feature in feature_names:
                    new_row[f'{feature}.{category}.Bern'] = np.nan
                for feature in feature_names:
                    new_row[f'{feature}.{category}.Poisson'] = np.nan
        
        rows.append(new_row)
    
    df = pd.DataFrame(rows, columns=non_stat_zip_model_columns)
    
    # Fill NaN values with column averages
    print("Filling missing parameters with averages...")
    for col in non_stat_zip_model_columns[1:]:  # Skip 'User' column
        if df[col].isna().any():
            avg_value = df[col].mean()
            num_missing = df[col].isna().sum()
            df[col] = df[col].fillna(avg_value)
            print(f"  {col}: filled {num_missing} missing values with average {avg_value:.4f}")
    
    return df

non_stat_zip_df = fill_missing_parameters_with_averages(zip_model_params, non_stat_zip_model_columns, feature_names, message_categories)

Filling missing parameters with averages...
  state_tod.QA.Bern: filled 1 missing values with average -0.0128
  state_b_bar.norm.QA.Bern: filled 1 missing values with average 0.0004
  state_a_bar.norm.QA.Bern: filled 1 missing values with average 0.0310
  state_app_engage.QA.Bern: filled 1 missing values with average -0.0067
  state_day_type.QA.Bern: filled 1 missing values with average -0.0441
  state_bias.QA.Bern: filled 1 missing values with average 0.0127
  state_day_in_study.QA.Bern: filled 1 missing values with average -0.0738
  state_tod.QA.Poisson: filled 1 missing values with average 0.0079
  state_b_bar.norm.QA.Poisson: filled 1 missing values with average 0.0386
  state_a_bar.norm.QA.Poisson: filled 1 missing values with average -0.0724
  state_app_engage.QA.Poisson: filled 1 missing values with average 0.0227
  state_day_type.QA.Poisson: filled 1 missing values with average 0.0650
  state_bias.QA.Poisson: filled 1 missing values with average -0.0895
  state_day_in_study.QA.

In [ ]:
# # Check which users have fewer than 4 message type categories
# user_category_counts = {}

# for user_id in MRT_DATA_WITH_MESSAGE_TYPE['user_id'].unique():
#     user_data = MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user_id]
#     user_categories = [cat for cat in user_data['message_type_category'].unique() if pd.notna(cat)]
#     user_category_counts[user_id] = len(user_categories)

# # Convert to DataFrame for easier analysis
# category_analysis = pd.DataFrame([
#     {'user_id': user, 'num_categories': count, 'categories': [cat for cat in MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user]['message_type_category'].unique() if pd.notna(cat)]}
#     for user, count in user_category_counts.items()
# ])

# print("Message type category distribution:")
# print(category_analysis['num_categories'].value_counts().sort_index())

# print(f"\nUsers with fewer than 4 categories:")
# users_less_than_4 = category_analysis[category_analysis['num_categories'] < 4]
# print(f"Count: {len(users_less_than_4)}")

# if len(users_less_than_4) > 0:
#     print("\nDetails:")
#     for _, row in users_less_than_4.iterrows():
#         print(f"  {row['user_id']}: {row['num_categories']} categories - {row['categories']}")

# print(f"\nUsers with exactly 4 categories:")
# users_with_4 = category_analysis[category_analysis['num_categories'] == 4]
# print(f"Count: {len(users_with_4)}")

# print(f"\nUsers with more than 4 categories:")
# users_more_than_4 = category_analysis[category_analysis['num_categories'] > 4]
# print(f"Count: {len(users_more_than_4)}")


Message type category distribution:
num_categories
3     1
4    71
Name: count, dtype: int64

Users with fewer than 4 categories:
Count: 1

Details:
  robas+173@developers.pg.com: 3 categories - ['SR', 'RP', 'FB']

Users with exactly 4 categories:
Count: 71

Users with more than 4 categories:
Count: 0


In [29]:
non_stat_zip_df

,User,state_tod.Base.Bern,state_b_bar.norm.Base.Bern,state_a_bar.norm.Base.Bern,state_app_engage.Base.Bern,state_day_type.Base.Bern,state_bias.Base.Bern,state_day_in_study.Base.Bern,state_tod.Base.Poisson,state_b_bar.norm.Base.Poisson,...,state_day_type.FB.Bern,state_bias.FB.Bern,state_day_in_study.FB.Bern,state_tod.FB.Poisson,state_b_bar.norm.FB.Poisson,state_a_bar.norm.FB.Poisson,state_app_engage.FB.Poisson,state_day_type.FB.Poisson,state_bias.FB.Poisson,state_day_in_study.FB.Poisson
0,robas+119@developers.pg.com,0.638660,-0.410344,-0.588092,0.023418,0.345855,-0.447821,0.510891,-0.947913,-0.147733,...,0.073915,-0.167166,-0.140999,-0.007808,-0.115978,-0.084931,0.252149,-0.000399,-0.007753,0.312995
1,robas+126@developers.pg.com,1.823311,0.093703,0.122664,-0.328100,-0.155351,-0.722057,0.046942,-0.709547,-0.105508,...,0.328929,-0.540814,0.612718,0.137178,-0.078188,0.034872,0.136643,-0.000770,0.137284,-0.001593
2,digitaldentalcoach+214@gmail.com,-0.038249,0.469690,0.590154,0.313672,0.305545,-0.946978,-0.086539,-0.001403,0.009143,...,-0.245603,0.240168,0.452590,0.301897,1.123671,-0.161696,-0.495213,0.002802,0.306475,0.026283
3,robas+135@developers.pg.com,0.416388,0.927687,0.167605,-0.065903,-0.110458,-1.126317,-0.095721,0.296350,-0.098857,...,0.064693,-0.026587,-0.000795,0.000326,-0.347109,0.007816,-0.038696,0.089098,-0.030251,0.071166
4,robas+199@developers.pg.com,2.647370,-1.356796,0.658291,0.076880,-0.279152,-1.226675,1.835105,0.023958,-0.001476,...,1.287036,-0.347348,-0.503880,0.000177,-0.247909,-0.047081,-0.169216,-0.438453,0.276383,0.377136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,digitaldentalcoach+249@gmail.com,0.731281,0.633216,0.762724,-0.257405,0.206908,-0.507414,-0.377950,0.037770,-0.214278,...,0.165270,0.232825,0.266306,-0.065565,-0.235473,1.011730,-0.189779,0.337487,0.566894,0.450754
68,digitaldentalcoach+271@gmail.com,0.922341,-0.064801,-0.045861,-0.352171,0.348789,-1.147005,-0.347702,0.565000,0.531220,...,-0.000539,-0.000288,-0.000652,-0.001384,-0.207771,0.571982,-0.763744,0.000371,0.292683,0.994774
69,digitaldentalcoach+236@gmail.com,1.807425,-0.947854,0.619578,-0.109509,0.206064,-1.320078,-0.504213,0.658348,-0.129877,...,0.267673,-0.099384,-0.108606,-0.001195,0.530469,1.960727,-0.067021,0.557630,0.489700,0.108291
70,robas+118@developers.pg.com,0.633004,-1.247932,-0.875026,0.442194,0.123902,-0.647892,-0.408855,0.427561,-0.091405,...,0.610353,-0.515191,-1.765205,-0.000610,-0.105668,0.521501,-0.000053,0.000366,0.144486,0.196337


## Saving to CSV
---

In [33]:
output_path = os.path.join(script_dir, '../../sim_env_data/v4_non_stat_zip_model_params.csv')
non_stat_zip_df.to_csv(output_path)